# 02 — Signal Processing and ERD/ERS Topographies

**Goal:** extract task-specific epochs and demonstrate that the post-stroke cortex still exhibits contralateral event-related desynchronization (ERD) during motor imagery — the physiological signal every downstream classifier depends on.

Mu (8-12 Hz) / beta (15-30 Hz) power drops over the contralateral sensorimotor cortex during imagined movement (C3 for right hand, C4 for left hand), rebounding (ERS) at movement cessation.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import mne

from src.data_loader import get_dataset, get_paradigm, FMIN, FMAX, TMIN, TMAX

## 1. Define the MOABB MotorImagery paradigm (8-30 Hz, 0-4 s post-cue)

In [ ]:
dataset = get_dataset()
paradigm = get_paradigm(fmin=FMIN, fmax=FMAX, tmin=TMIN, tmax=TMAX)
print(paradigm)

## 2. Isolate left-hand and right-hand epochs for one subject

In [ ]:
subject_id = dataset.subject_list[0]
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[subject_id], return_epochs=True)

epochs = X  # return_epochs=True gives an mne.Epochs object
print(epochs)
print('Event counts:', {k: len(epochs[k]) for k in epochs.event_id})

## 3. Time-Frequency Representation via Morlet wavelets (6-35 Hz)

Computed separately for left-hand and right-hand trials so the contralateral asymmetry is visible.

In [ ]:
freqs = np.arange(6, 36, 1)
n_cycles = freqs / 2.0

left_key = [k for k in epochs.event_id if 'left' in k.lower()][0]
right_key = [k for k in epochs.event_id if 'right' in k.lower()][0]

power_left = epochs[left_key].compute_tfr(
    method='morlet', freqs=freqs, n_cycles=n_cycles, use_fft=True,
    return_itc=False, average=True, decim=3,
)
power_right = epochs[right_key].compute_tfr(
    method='morlet', freqs=freqs, n_cycles=n_cycles, use_fft=True,
    return_itc=False, average=True, decim=3,
)

## 4. Topographical ERD/ERS maps — contralateral mu suppression

In [ ]:
baseline = (TMIN, TMIN + 0.5)  # short pre-imagery baseline window

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
power_left.plot_topomap(
    tmin=1.0, tmax=3.0, fmin=8, fmax=12, baseline=baseline, mode='percent',
    axes=axes[0], show=False, colorbar=True,
)
axes[0].set_title(f'Left-hand MI — mu ERD\n(expect suppression over C4)')

power_right.plot_topomap(
    tmin=1.0, tmax=3.0, fmin=8, fmax=12, baseline=baseline, mode='percent',
    axes=axes[1], show=False, colorbar=True,
)
axes[1].set_title(f'Right-hand MI — mu ERD\n(expect suppression over C3)')
plt.tight_layout()
plt.show()

## Interpretation checklist
- Left-hand imagery → suppression should center over **C4** (right hemisphere).
- Right-hand imagery → suppression should center over **C3** (left hemisphere).
- In acute stroke, expect this asymmetry to be **blunted, delayed, or spatially shifted** relative to healthy cohorts (Section 2.2, "Pathological Variability") — note it here per subject rather than assuming a clean textbook pattern.

## Next
Proceed to **`03_Spatial_Filtering_CSP.ipynb`** to derive CSP spatial filters that exploit exactly this contralateral variance difference.